In [1]:
%load_ext autoreload
%autoreload 2

In [21]:
import numpy as np
import pandas as pd
import plotly.express as px
from pathlib import Path

In [ ]:
weather_dirpath = Path('../../data/raw/weather_owm')
coords = []
for filepath in weather_dirpath.glob('*.json'):
    parts = filepath.stem.split('_')
    lat = float('.'.join(parts[1:3]))
    lon = float('.'.join(parts[3:5]))
    coords.append((lat, lon))
coords

[(51.6956, 75.3297),
 (47.247706, 52.465977),
 (46.862232, 73.500071),
 (47.885021, 68.555921),
 (50.333435, 57.162802),
 (53.257756, 63.701736),
 (53.221389, 64.580278),
 (50.397318, 81.069005),
 (51.234698, 51.405632),
 (49.980758, 82.615375),
 (42.845057, 71.480839),
 (52.406688, 64.082731),
 (42.161097, 69.761446),
 (51.0271, 71.4607),
 (51.2033, 61.4431),
 (45.495101, 64.114344),
 (42.859331, 73.211756),
 (44.18764, 66.719456),
 (43.476747, 73.647966),
 (48.608459, 57.109526),
 (43.033564, 78.83436),
 (49.921055, 83.085657),
 (50.271911, 127.49593),
 (43.622656, 78.596191),
 (41.25273, 67.967631),
 (52.211445, 76.911346),
 (51.627527, 69.836461),
 (46.776875, 54.532018),
 (43.610907, 76.962575),
 (51.267981, 51.096781),
 (45.086392, 51.387305),
 (43.082778, 76.976111),
 (46.732508, 49.735043),
 (50.063887, 72.986541),
 (50.188422, 51.165903),
 (49.802287, 72.864697),
 (52.164818, 61.269958),
 (42.432744, 68.813798),
 (48.548004, 57.564817),
 (50.513153, 58.261809),
 (47.675556, 51

In [35]:
# Sample data with latitude and longitude
df = pd.DataFrame(coords, columns=['Latitude', 'Longitude'])
df['type'] = 'weather_station'

# Create a geographical scatter plot
fig = px.scatter_geo(df, 
                        lat='Latitude',
                        lon='Longitude',
                     title='Weather Station Coordinates',)
fig.update_geos(
    showcountries=True, countrycolor="Black",
    showsubunits=True, subunitcolor="Brown",
    showland=True, landcolor="LightGreen",
    showocean=True, oceancolor="LightBlue",
    showframe=True, framecolor="DarkGrey",
    resolution=50 # or 110
)
fig.show()

In [36]:
DEFAULT_LAT = 43.256700
DEFAULT_LON = 76.928600

dfs = []
for filepath in Path('../../data/raw/ss_coords').glob('*.xlsx'):
    mes_name = filepath.stem
    df_ss = pd.read_excel(filepath)
    mask = \
        (df_ss['lat'] != ':atom:null') & \
        (df_ss['lon'] != ':atom:null') 
    df_ss = df_ss[mask]
    df_ss['lat'] = df_ss['lat'].astype(float)
    df_ss['lon'] = df_ss['lon'].astype(float)
    mask = ~((np.isclose(df_ss['lat'], DEFAULT_LAT, atol=1e-2)) & (np.isclose(df_ss['lon'], DEFAULT_LON, atol=1e-2))) & \
        (df_ss['lat'] >= 40.66667) & (df_ss['lat'] <= 54.90521) & \
        (df_ss['lon'] >= 46.84705) & (df_ss['lon'] <= 84.87144) & \
        (df_ss['lat'].notna()) & (df_ss['lon'].notna())
    df_ss = df_ss[mask]
    df_ss = df_ss[['prototype@weather', 'lat', 'lon']]
    df_ss['prototype@weather'] = '/' + df_ss['prototype@weather'].str.split('/').str[2:-1].str.join('/')

    # Add "/{mes_name}" row with default coordinates
    # calculate them as mean cooridnates of \
    # all other stations except ones that has "@regions" in their names
    non_region_mask = ~df_ss['prototype@weather'].str.contains('@regions')
    df_ss = pd.concat([
        df_ss,
        pd.DataFrame([{
            'prototype@weather': f'/{mes_name}',
            'lat': df_ss.loc[non_region_mask, 'lat'].mean(),
            'lon': df_ss.loc[non_region_mask, 'lon'].mean(),
        }])
    ], ignore_index=True)

    dfs.append(df_ss)
df_all_ss = pd.concat(dfs, ignore_index=True)
display(df_all_ss[df_all_ss.duplicated(subset=['prototype@weather'], keep=False)].sort_values(by='prototype@weather'))
# Manually set 
# /SEVER
# /ALMATY
# /VOSTOK
# /CENTER
# /KOSTANAY
df_all_ss['type'] = 'substation'
df_all_ss

,prototype@weather,lat,lon


,prototype@weather,lat,lon,type
0,/AKTOBE/Zhazyk,51.303995,51.209345,substation
1,/AKTOBE/Uzlovaya,51.446499,51.265491,substation
2,/AKTOBE/Ural_GTES,51.483871,51.272455,substation
3,/AKTOBE/U_TEC,51.423282,51.222241,substation
4,/AKTOBE/Karachaganak,53.389608,51.450781,substation
5,/AKTOBE/GTES-200,51.099005,51.266977,substation
6,/AKTOBE/Chapaevo,51.131609,50.873127,substation
7,/AKTOBE/@regions/Aktobe,50.283900,57.167000,substation
8,/AKTOBE,51.611124,51.222917,substation
9,/SEVER,NaN,NaN,substation


In [38]:
df_merged = pd.concat(
    [
        df_all_ss[['type', 'lat', 'lon']],
        df[['type', 'Latitude', 'Longitude']].rename(columns={
            'Latitude': 'lat',
            'Longitude': 'lon'
        }),
    ],
    ignore_index=True
)
df_merged

,type,lat,lon
0,substation,51.303995,51.209345
1,substation,51.446499,51.265491
2,substation,51.483871,51.272455
3,substation,51.423282,51.222241
4,substation,53.389608,51.450781
...,...,...,...
207,weather_station,44.722180,65.893614
208,weather_station,52.639607,70.452747
209,weather_station,43.377531,76.730231
210,weather_station,48.257747,70.211460


In [42]:
# Create a geographical scatter plot
fig = px.scatter_geo(df_merged, 
                        lat='lat',
                        lon='lon',
                     title='Coordinates',
                     color='type',
                     symbol='type',
                    
                     )
fig.update_geos(
    showcountries=True, countrycolor="Black",
    showsubunits=True, subunitcolor="Brown",
    showland=True, landcolor="LightGreen",
    showocean=True, oceancolor="LightBlue",
    showframe=True, framecolor="DarkGrey",
    resolution=50 # or 110
)
fig.show()